In [ ]:

cd "D:\ai project\WasteWise-AI"

New-Item -ItemType Directory -Force -Path ".\ml\src" | Out-Null

@'
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path(__file__).resolve().parents[2]
RAW_DIR = BASE_DIR / "ml" / "data" / "raw"
OUTPUT_DIR = BASE_DIR / "ml" / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_file = RAW_DIR / "perishable_inventory.csv"
output_file = OUTPUT_DIR / "featured_inventory.csv"

if not input_file.exists():
    raise FileNotFoundError(f"Dataset not found: {input_file}")

df = pd.read_csv(input_file)

df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df["available_stock"] = df["opening_stock"] + df["units_received"]

df["remaining_stock"] = (
    df["available_stock"] - df["units_sold"]
).clip(lower=0)

df["sales_rate"] = np.where(
    df["available_stock"] > 0,
    df["units_sold"] / df["available_stock"],
    0
)

df["discount_amount_pct"] = df["discount_pct"].fillna(0)

df["expiry_risk"] = pd.cut(
    df["days_to_expiry"],
    bins=[-np.inf, 2, 5, 10, np.inf],
    labels=["Critical", "High", "Medium", "Low"]
)

df["temperature_risk"] = pd.cut(
    df["temperature_c"],
    bins=[-np.inf, 5, 15, 25, np.inf],
    labels=["Cold", "Cool", "Warm", "Hot"]
)

df["waste_occurred"] = (df["waste_units"] > 0).astype(int)

df = df.drop_duplicates()
df = df.replace([np.inf, -np.inf], np.nan)

df.to_csv(output_file, index=False)

print("FEATURE ENGINEERING COMPLETED SUCCESSFULLY")
print("Dataset shape:", df.shape)
print("\nNew features:")
print([
    "year",
    "month",
    "day",
    "day_of_week",
    "is_weekend",
    "available_stock",
    "remaining_stock",
    "sales_rate",
    "expiry_risk",
    "temperature_risk",
    "waste_occurred"
])
print("\nMissing values:")
print(df.isnull().sum())
print("\nOutput saved to:", output_file)
print("\nFirst five rows:")
print(df.head())
'@ | Set-Content -Encoding UTF8 ".\ml\src\feature_engineering.py"
